# NeuroScan MRI - Brain Tumor Classification
This notebook trains a **ResNet50** model to classify MRI scans into four categories:
1. Glioma
2. Meningioma
3. No Tumor
4. Pituitary

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import zipfile
import tarfile
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ==================== Configuration ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 0.001
DATA_DIR = Path("./Brain_MRI")
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']

## 1. Extract Dataset
Upload your compressed dataset to the SageMaker instance first, then run the appropriate cell below.

In [ ]:
# REPLACE 'dataset.zip' with your actual filename
zip_filename = 'dataset.zip' 

if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('./Brain_MRI')
    print("Dataset extracted successfully!")
else:
    print(f"File {zip_filename} not found. Please upload it first.")

## 2. Preprocessing & Data Loaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BrainMRIDataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        split_dir = 'Training' if split.lower() == 'train' else 'Testing'
        self.data_dir = Path(data_dir) / split_dir
        self.transform = transform
        self.classes = CLASSES
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.images, self.labels = [], []
        
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.images.append(str(img_path))
                    self.labels.append(self.class_to_idx[class_name])
    
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, self.labels[idx]

## 3. Training Loop

In [ ]:
train_dataset = BrainMRIDataset(DATA_DIR, split='train', transform=train_transform)
test_dataset = BrainMRIDataset(DATA_DIR, split='test', transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return running_loss / len(loader), 100 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return running_loss / len(loader), 100 * correct / total

best_val_acc = 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train Acc {train_acc:.2f}%, Val Acc {val_acc:.2f}%")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
print(f"Best Accuracy: {best_val_acc:.2f}%")

## 4. Package & Deploy to AWS
Run these cells to create the live API Endpoint for your web app.

In [ ]:
%%writefile inference.py
import os, torch, torch.nn as nn, json, io
from torchvision import models, transforms
from PIL import Image
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
def model_fn(model_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    with open(os.path.join(model_dir, "model.pth"), "rb") as f:
        model.load_state_dict(torch.load(f, map_location=device))
    return model.to(device).eval()
def input_fn(body, content_type):
    return Image.open(io.BytesIO(body)).convert("RGB")
def predict_fn(input_data, model):
    tr = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    input_tensor = tr(input_data).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)
        confidence, predicted = torch.max(torch.softmax(output, dim=1), 1)
    return {"result": CLASSES[predicted.item()], "confidence": confidence.item()}
def output_fn(prediction, content_type): return json.dumps(prediction)

In [ ]:
import sagemaker, tarfile
from sagemaker.pytorch import PyTorchModel
from sagemaker.serverless import ServerlessInferenceConfig

# 1. Package model
os.rename('best_model.pth', 'model.pth')
with tarfile.open('model.tar.gz', 'w:gz') as tar: tar.add('model.pth')

# 2. Deploy
sess = sagemaker.Session()
s3_path = sess.upload_data('model.tar.gz', bucket=sess.default_bucket(), key_prefix='model')
pytorch_model = PyTorchModel(model_data=s3_path, role=sagemaker.get_execution_role(), entry_point='inference.py', framework_version='2.0.1', py_version='py310')
serverless_config = ServerlessInferenceConfig(memory_size_in_mb=3072, max_concurrency=1)
predictor = pytorch_model.deploy(serverless_inference_config=serverless_config)
print(f"Endpoint Name: {predictor.endpoint_name}")